In [ ]:
"""
To run this test, download the OPSD dataset folder from 
https://data.open-power-system-data.org/time_series/
and place the 'opsd-time_series-2020-10-06' folder as a top-level subfolder in 
datasets_and_dataloaders. Then run this script.
"""

import os
import sys
import numpy as np
import pandas as pd
import pyscamp

import random
import importlib

current_dir = os.getcwd()
sys.path.append(current_dir)
parent_dir = os.path.dirname(current_dir)
sys.path.append(parent_dir)

from datasets_and_dataloaders.dataloader import load_opsd_data


import mplot_python.MINT as M
importlib.reload(M)

from mplot_python.MINT import processAll, nnrobustpca_stable_pcp


In [21]:
data_path = os.path.join(parent_dir, 'datasets_and_dataloaders', 'opsd-time_series-2020-10-06')

df = load_opsd_data(data_path=data_path)
print(df.head(30))
print(df.tail())

limit = 0  # threshold for switching to random fill

for column in df.columns:
    if column == "utc_timestamp":
        continue

    series = df[column].copy()
    mask = series.isna()

    # --- Identify NaN runs ---
    group = (mask != mask.shift()).cumsum()
    nan_groups = series[mask].groupby(group[mask])

    # Compute column-wide mean and std for random filling
    mean = np.nanmean(series)
    sigma = np.nanstd(series)
    rand_background = np.random.normal(mean, sigma, len(series))

    for g, idxs in nan_groups.groups.items():
        run_length = len(idxs)
        start = idxs[0]
        end = idxs[-1]

        # --- short run (< limit) -> deterministic interpolation ---
        if run_length < limit:
            print(f"Imputing {run_length} NaNs in column '{column}' at positions {start}-{end} with interpolation.")
            prev_idx = start - 1 if start > 0 else None
            next_idx = end + 1 if end + 1 < len(series) else None

            if prev_idx is not None and next_idx is not None and not np.isnan(series[prev_idx]) and not np.isnan(series[next_idx]):
                interp_vals = np.linspace(series[prev_idx], series[next_idx], run_length + 2)[1:-1]
                series.iloc[idxs] = interp_vals
            else:
                 #  fill with mean if one side missing
                series.iloc[idxs] = mean

        # --- long run (>= limit) -> random Gaussian fill ---
        else:
            print(f"Imputing {run_length} NaNs in column '{column}' at positions {start}-{end} with random Gaussian fill.")

            series.iloc[idxs] = rand_background[idxs]

    df[column] = series

Looking for files in: c:\Users\masia\mint-decomposition\datasets_and_dataloaders\opsd-time_series-2020-10-06
['datapackage.json', 'README.md', 'time_series.xlsx', 'time_series_15min_singleindex.csv', 'time_series_30min_singleindex.csv', 'time_series_60min_singleindex.csv']
Loaded OPSD dataset with shape: (50401, 300)
           utc_timestamp  Austria  Cyprus  Germany  Denmark  Estonia    Spain  \
0   2014-12-31T23:00:00Z      NaN     NaN      NaN      NaN      NaN      NaN   
1   2015-01-01T00:00:00Z   5946.0     NaN  41151.0      NaN      NaN      NaN   
2   2015-01-01T01:00:00Z   5726.0     NaN  40135.0  3100.02    764.7  22734.0   
3   2015-01-01T02:00:00Z   5347.0     NaN  39106.0  2980.39    749.8  21286.0   
4   2015-01-01T03:00:00Z   5249.0     NaN  38765.0  2933.49    746.5  20264.0   
5   2015-01-01T04:00:00Z   5309.0     NaN  38941.0  2941.54    754.6  19905.0   
6   2015-01-01T05:00:00Z   5574.0     NaN  39045.0  2999.89    771.2  20010.0   
7   2015-01-01T06:00:00Z   5925.0

In [22]:
len(df) / 167

301.8023952095808

In [23]:
import numpy as np

np.random.seed(158)

subsequenceLength = 167
col_list = [c for c in df.columns if c != "utc_timestamp"]
exclude = []

print("filtering sensors...")
for idx, sensor_name in enumerate(col_list):

    print("Processing sensor:", sensor_name)
    series = df[sensor_name].to_numpy().astype(np.float32)    

    mplot = pyscamp.abjoin_matrix(
        series, series, subsequenceLength,
        mheight=418, mwidth=418, threshold=-1
    )

    if np.isnan(mplot).any():
        print(sensor_name + " excluded due to NaNs in MATRIX PROFILE")
        exclude.append(idx)
        
col_list = [col for i, col in enumerate(col_list) if i not in exclude]



results =  processAll(col_list, df, subsequenceLength, Mheight = 300, Mwidth = 300, name = "OPSD")

filtering sensors...
Processing sensor: Austria
Processing sensor: Cyprus
Processing sensor: Germany
Processing sensor: Denmark
Processing sensor: Estonia
Processing sensor: Spain
Processing sensor: Great Britain
Processing sensor: United Kingdom
Processing sensor: Greece
Processing sensor: Croatia
Processing sensor: Hungary
Processing sensor: Italy
Processing sensor: Lithuania
Processing sensor: Latvia
Processing sensor: Norway
Processing sensor: Portugal
Processing sensor: Sweden
Processing sensor: Slovakia
===TENSOR CALCULATION===
calculating mplots...
0 Austria
beginning RPCA...
RPCA done! Converged in 87 iteration(s).
tensor(1112.0943, dtype=torch.float64)
tensor(73.7522, dtype=torch.float64)
1 Cyprus
beginning RPCA...
RPCA done! Converged in 22 iteration(s).
tensor(3803.7740, dtype=torch.float64)
tensor(165.2588, dtype=torch.float64)
2 Germany
beginning RPCA...
RPCA done! Converged in 116 iteration(s).
tensor(822.5064, dtype=torch.float64)
tensor(66.2224, dtype=torch.float64)
3 D

In [24]:
from mplot_python.MINT import plotMatrixRaw
import numpy as np

A,B,C = results.low_rank_factors

A = np.abs(A)
B = np.abs(B)
C = np.abs(C)

color_map = "viridis"

plotMatrixRaw(A, "OPSD_A", "OPSDj", colormap = color_map)
plotMatrixRaw(B, "OPSD_B", "OPSDj", colormap = color_map)
plotMatrixRaw(C, "OPSD_C", "OPSDj", colormap = color_map)

In [25]:
pd.to_datetime(df.loc[0, "utc_timestamp"]) + pd.Timedelta(days = 7*90)

Timestamp('2016-09-21 23:00:00+0000', tz='UTC')

In [26]:
component_indices = range(C.shape[1])
s = 5
for i in component_indices:
    print(f"Component {i}")
    component = C[:, i]
    component_abs = np.abs(component)

    # Top-s indices in descending order
    ind = np.argpartition(component_abs, -s)[-s:]
    top_s_indices = ind[np.argsort(component_abs[ind])]
    top_s_indices = np.flip(top_s_indices)
    
    for j in top_s_indices:
        print(j, pd.to_datetime(df.loc[0, "utc_timestamp"]) + pd.Timedelta(days = 7*j), C[j,i])

Component 0
147 2017-10-25 23:00:00+00:00 0.08173181
251 2019-10-23 23:00:00+00:00 0.08011475
253 2019-11-06 23:00:00+00:00 0.07977488
145 2017-10-11 23:00:00+00:00 0.0796743
201 2018-11-07 23:00:00+00:00 0.07962896
Component 1
51 2015-12-23 23:00:00+00:00 0.06878049
101 2016-12-07 23:00:00+00:00 0.0673352
147 2017-10-25 23:00:00+00:00 0.06701339
260 2019-12-25 23:00:00+00:00 0.06691337
0 2014-12-31 23:00:00+00:00 0.06634185
Component 2
150 2017-11-15 23:00:00+00:00 0.09580401
106 2017-01-11 23:00:00+00:00 0.0955604
214 2019-02-06 23:00:00+00:00 0.095124796
160 2018-01-24 23:00:00+00:00 0.094977096
98 2016-11-16 23:00:00+00:00 0.094267406
Component 3
192 2018-09-05 23:00:00+00:00 0.07379054
191 2018-08-29 23:00:00+00:00 0.073178984
139 2017-08-30 23:00:00+00:00 0.07307891
89 2016-09-14 23:00:00+00:00 0.0722086
88 2016-09-07 23:00:00+00:00 0.07137206


In [27]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd

start = pd.to_datetime(df.loc[0, "utc_timestamp"])
x = start + pd.to_timedelta([7 * i for i in range(C.shape[0])], unit="D")

fig = make_subplots(
    rows=len(component_indices),
    cols=1,
    shared_xaxes=True,
    subplot_titles=[f"Component {i + 1}" for i in component_indices],
    vertical_spacing=0.10,
)

for row, i in enumerate(component_indices, start=1):
    fig.add_trace(
        go.Scatter(
            x=x,
            y=C[:, i],
            mode="lines",
            showlegend=False,
        ),
        row=row,
        col=1,
    )

# Show date labels on every subplot
fig.update_xaxes(showticklabels=True)


fig.update_layout(
    height=250 * len(component_indices),
    template="plotly_white",
)

fig.update_layout(margin=dict(l=90))

fig.show()

In [28]:
print(pd.to_datetime(df.loc[0, "utc_timestamp"]) + pd.Timedelta(days = 7*88))
print(pd.to_datetime(df.loc[0, "utc_timestamp"]) + pd.Timedelta(days = 7*98))
print(pd.to_datetime(df.loc[0, "utc_timestamp"]) + pd.Timedelta(days = 7*103))
print(pd.to_datetime(df.loc[0, "utc_timestamp"]) + pd.Timedelta(days = 7*106))
print(pd.to_datetime(df.loc[0, "utc_timestamp"]) + pd.Timedelta(days = 7*106))
print(pd.to_datetime(df.loc[0, "utc_timestamp"]) + pd.Timedelta(days = 7*117))

2016-09-07 23:00:00+00:00
2016-11-16 23:00:00+00:00
2016-12-21 23:00:00+00:00
2017-01-11 23:00:00+00:00
2017-01-11 23:00:00+00:00
2017-03-29 23:00:00+00:00


In [29]:
from mplot_python.co_clustering_trial import create_processed_dataframe

b_prop = 0.15
num_of_windows = 45
window_size = 167


processed_dataframe, chosen_intervals, completely_random, mostly_random, mostly_normal = create_processed_dataframe(df, col_list, b_prop, num_of_windows, window_size)




In [30]:
print(len(processed_dataframe))

50401


In [31]:
print(chosen_intervals)
print(completely_random)
print(mostly_random)
print(mostly_normal)

[[20374, 20540], [668, 834], [20708, 20874], [10020, 10186], [33400, 33566], [44088, 44254], [14028, 14194], [2839, 3005], [11857, 12023], [50100, 50266], [14529, 14695], [15698, 15864], [25551, 25717], [41583, 41749], [43420, 43586], [47261, 47427], [23213, 23379], [501, 667], [40247, 40413], [35404, 35570], [34903, 35069], [15364, 15530], [41917, 42083], [10187, 10353], [4342, 4508], [28223, 28389], [49098, 49264], [25217, 25383], [30227, 30393], [40581, 40747], [28390, 28556], [33567, 33733], [4175, 4341], [4008, 4174], [36406, 36572], [48430, 48596], [39078, 39244], [33233, 33399], [0, 166], [23380, 23546], [45591, 45757], [18537, 18703], [7014, 7180], [43754, 43920], [14362, 14528]]
['Italy', 'Estonia', 'Great Britain']
['Croatia', 'Denmark', 'Portugal']
['Sweden', 'Spain', 'United Kingdom', 'Greece', 'Hungary', 'Germany', 'Norway', 'Lithuania', 'Austria', 'Slovakia', 'Cyprus', 'Latvia']


In [ ]:
rand = processed_dataframe["Germany"].to_numpy()
rand_mplot = pyscamp.abjoin_matrix(
                np.copy(rand),  # Convert to Python list
                np.copy(rand),  # Convert to Python list
                subsequenceLength, 
                mheight=300, 
                mwidth=300, 
                threshold=-1
            )

plotMatrixRaw(rand_mplot, "OPSD_rand_M", "OPSD", colormap = color_map)




In [33]:
print(num_iter)

1086


In [34]:
rand_sL, rand_sS, num_iter = nnrobustpca_stable_pcp(rand_mplot, max_iter = 5000)

plotMatrixRaw(rand_sL, "OPSD_rand_sL", "OPSD", colormap = color_map)
plotMatrixRaw(rand_sS, "OPSD_rand_sS", "OPSD", colormap = color_map)

In [ ]:
norm = processed_dataframe["Portugal"].to_numpy()
norm_mplot = pyscamp.abjoin_matrix(
                np.copy(norm),  # Convert to Python list
                np.copy(norm),  # Convert to Python list
                subsequenceLength, 
                mheight=300, 
                mwidth=300, 
                threshold=-1
            )

plotMatrixRaw(norm_mplot, "OPSD_norm_M", "OPSD", colormap = color_map)




In [36]:
print(len(processed_dataframe))

50401


In [37]:
norm_sL, norm_sS, num_iter = nnrobustpca_stable_pcp(norm_mplot, max_iter = 5000)

plotMatrixRaw(norm_sL, "OPSD_norm_sL", "OPSD", colormap = color_map)
plotMatrixRaw(norm_sS, "OPSD_norm_sS", "OPSD", colormap = color_map)